In [ ]:
import os
from tifffile import tifffile
import numpy as np
import albumentations as A
import cv2
from tqdm import tqdm
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

def normalize_clip(img, vmin, vmax):
    img = np.clip(img, vmin, vmax)
    return (2 * (img - vmin) / (vmax - vmin) - 1)

def standardization(img):
    return (img - img.mean()) / img.std()

def stand_norm(img, img_minmax):
    img = standardization(img)
    return normalize_clip(img, img_minmax[0], img_minmax[1])

def gauss_clip_con(img, img_minmax, sigma=7, gain=4.0, biases=[0.1, 0.7]):
    vmin = img_minmax[0]
    vmax = img_minmax[1]
    img = np.clip((img - img.mean()) / img.std(), vmin, vmax)
    img = (img - vmin) / (vmax - vmin)
    bias_0 = gauss_clip(img, sigma=sigma, gain=gain, bias=biases[0]) == 0
    bias_1 = gauss_clip(img, sigma=sigma, gain=gain, bias=biases[1]) == 0
    img_bias = bias_0 * bias_1
    img = 2 * img - 1
    img[img_bias] = -1.0
    # img[img_bias] = 0.0
    return img.astype(np.float32)

def gauss_clip(img, sigma=5, gain=4.0, bias=0.2):
    img_sig = 1 / (1 + np.exp(-(gain * (img - bias))))
    gauss = gaussian_filter(img_sig, sigma=sigma)
    img_gauss = img_sig - gauss
    mask_border = np.percentile(img_gauss, 95)
    img_gauss_clip = img_sig.copy()
    img_gauss_clip[img_gauss < mask_border] = 0
    return img_gauss_clip

def produce_images(original_dir, main_dir, data_folders=["1", "2", "3", "4", "5"], train_idx=[1, 2, 3], val_test_idx=[0, 4],
                   kmeans=False, preprocess=True, img_n=100, img_size=256, gaussian_crip=False):
    original_img_folders = [os.path.join(original_dir, f) for f in data_folders]
    if preprocess:
        img_list = []
        for i in range(len(original_img_folders)):
            img_path_list = os.listdir(original_img_folders[i])
            for j in range(len(img_path_list)):
                image = tifffile.imread(os.path.join(original_img_folders[i], img_path_list[j]))
                img = []
                for l in range(len(image[0][0])):
                    img.append(standardization(image[..., l]).flatten())
                img_list.append([img])
        img_list = np.array(img_list)
        img_minmax = []
        for l in range(len(image[0][0])):
            img_concat = np.concatenate(img_list[:, 0, l, :])
            img_minmax.append([np.percentile(img_concat, 0.1), np.percentile(img_concat, 99.9)])
        print("norm_measure done")
    
    # 最終的な画像サイズ
    final_size = img_size
    # 最初に切り出す、一回り大きいサイズ
    initial_crop_size = img_size * 2 
    augment = A.Compose(
        [
            # 1. 元画像から大きめにクロップ (128x128)
            A.RandomCrop(height=initial_crop_size, width=initial_crop_size, border_mode=cv2.BORDER_REFLECT_101),
            # 2. そのパッチに対して回転・拡縮
            #    ※拡縮しすぎると中央に黒い部分が入る可能性があるので、scale_limitは控えめに
            A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=30, p=1.0),
            # 3. 変換されたパッチの中央から最終サイズをクロップ (64x64)
            A.CenterCrop(height=final_size, width=final_size),
        ],
        additional_targets={"image1": "image",
                            "image2": "image"},
        strict=True,
        # seed=137,
    )
    
    os.makedirs(main_dir, exist_ok=True)
    img_folders = [os.path.join(main_dir, f) for f in data_folders]
    for folder in img_folders:
        os.makedirs(folder, exist_ok=True)
    if kmeans:
        img_folders_cut = [os.path.join(main_dir, f"c{f}") for f in data_folders]
        for folder in img_folders_cut:
            os.makedirs(folder, exist_ok=True)
        
    for i in range(len(original_img_folders)):
        img_path_list = os.listdir(original_img_folders[i])
        for j in tqdm(range(len(img_path_list))):
            ori_pth = os.path.join(original_img_folders[i],img_path_list[j])
            pth = os.path.join(img_folders[i], img_path_list[j])
            image = tifffile.imread(ori_pth)
            if preprocess:
                for l in range(len(image[0][0])):
                    if l != 2:
                        image[..., l] = stand_norm(image[..., l], img_minmax[l])
                    else:
                        if gaussian_crip:
                            image[..., 2] = gauss_clip_con(image[..., 2], img_minmax[2])
                        else:
                            image[..., 2] = stand_norm(image[..., 2], img_minmax[2])
            phase1, phase2, mito = image[..., 0], image[..., 1], image[..., 2]
            if kmeans:
                tifffile.imwrite(pth, image)
            else:
                if i in val_test_idx:
                    tifffile.imwrite(pth, image)
            if kmeans or i in train_idx:
                for k in range(img_n):
                    augmented = augment(image=phase1, image1=phase2, image2=mito)
                    image_crop = np.stack([augmented['image'], augmented['image1'], augmented['image2']], axis=-1)
                    base_name, ext = os.path.splitext(img_path_list[j])
                    new_filename = f"{base_name}_{k}{ext}"
                    if kmeans:
                        save_path = os.path.join(img_folders_cut[i], new_filename)
                    else:
                        save_path = os.path.join(img_folders[i], new_filename)
                    tifffile.imwrite(save_path, image_crop)
        print(f"folder {img_folders[i]} done")

In [ ]:
produce_images(r"D:\Matsusaka\data_mito\HeLa_Su9-mSG",
               r"D:\Matsusaka\data_mito\mito_original_diffusion",
               data_folders=["1", "2", "3", "4", "5"],
               train_idx=[], val_test_idx=[0, 1, 2, 3, 4], kmeans=False, 
               preprocess=True, img_n=0, img_size=1, gaussian_crip=False,)

In [ ]:
produce_images(r"D:\Matsusaka\data_mito\HeLa_Su9-mSG",
               r"D:\Matsusaka\data_mito\mito_512_crop_diffusion",
               data_folders=["1", "2", "3", "4", "5"],
               train_idx=[1, 2, 3], val_test_idx=[0, 4], kmeans=False, 
               preprocess=True, img_n=30, img_size=512, gaussian_crip=False,)

In [ ]:
produce_images(r"D:\Matsusaka\data_mito\COS7_KDEL-mSG",
               r"D:\Matsusaka\data_mito\ER_128_crop_diffusion",
               data_folders=["1", "2", "3"],
               train_idx=[0], val_test_idx=[1, 2], kmeans=False, 
               preprocess=True, img_n=100, img_size=128, gaussian_crip=False,)

In [ ]:
# import os
# from tifffile import tifffile
# import numpy as np
# import albumentations as A
# import cv2
# from tqdm import tqdm
# from scipy.ndimage import gaussian_filter
# import matplotlib.pyplot as plt
# 
# def normalize_clip(img, vmin, vmax):
#     img = np.clip(img, vmin, vmax)
#     return (2 * (img - vmin) / (vmax - vmin) - 1)
# 
# def standardization(img):
#     return (img - img.mean()) / img.std()
# 
# def stand_norm(img, img_minmax):
#     img = standardization(img)
#     return normalize_clip(img, img_minmax[0], img_minmax[1])
# 
# def gauss_clip_con(img, img_minmax, sigma=7, gain=4.0, biases=[0.1, 0.7]):
#     vmin = img_minmax[0]
#     vmax = img_minmax[1]
#     img = np.clip((img - img.mean()) / img.std(), vmin, vmax)
#     img = (img - vmin) / (vmax - vmin)
#     bias_0 = gauss_clip(img, sigma=sigma, gain=gain, bias=biases[0]) == 0
#     bias_1 = gauss_clip(img, sigma=sigma, gain=gain, bias=biases[1]) == 0
#     img_bias = bias_0 * bias_1
#     img = 2 * img - 1
#     img[img_bias] = -1.0
#     # img[img_bias] = 0.0
#     return img.astype(np.float32)
# 
# def gauss_clip(img, sigma=5, gain=4.0, bias=0.2):
#     img_sig = 1 / (1 + np.exp(-(gain * (img - bias))))
#     gauss = gaussian_filter(img_sig, sigma=sigma)
#     img_gauss = img_sig - gauss
#     mask_border = np.percentile(img_gauss, 95)
#     img_gauss_clip = img_sig.copy()
#     img_gauss_clip[img_gauss < mask_border] = 0
#     return img_gauss_clip
        
# def produce_images(original_dir, main_dir, data_folders=["1", "2", "3", "4", "5"], train_idx=[1, 2, 3], val_test_idx=[0, 4],
#                    preprocess=True, img_n=100, img_size=256, kmeans=False):
#     if preprocess:
#         original_img_folders = [os.path.join(original_dir, f) for f in [data_folders[k] for k in train_idx]]
#         img_list = []
#         for i in range(len(original_img_folders)):
#             img_path_list = os.listdir(original_img_folders[i])
#             for j in range(len(img_path_list)):
#                 image = tifffile.imread(os.path.join(original_img_folders[i], img_path_list[j]))
#                 img = []
#                 for l in range(len(image[0][0])):
#                     img.append(standardization(image[..., l]).flatten())
#                 img_list.append([img])
#         img_list = np.array(img_list)
#         img_minmax = []
#         for l in range(len(image[0][0])):
#             img_concat = np.concatenate(img_list[:, 0, l, :])
#             img_minmax.append([np.percentile(img_concat, 0.1), np.percentile(img_concat, 99.9)])
#         print("norm_measure done")
#         
#     # 最終的な画像サイズ
#     final_size = img_size
#     # 最初に切り出す、一回り大きいサイズ
#     initial_crop_size = img_size * 2 
#     
#     train_aug = A.Compose(
#         [
#             # 1. 元画像から大きめにクロップ (128x128)
#             A.RandomCrop(height=initial_crop_size, width=initial_crop_size),
#             # 2. そのパッチに対して回転・拡縮
#             #    ※拡縮しすぎると中央に黒い部分が入る可能性があるので、scale_limitは控えめに
#             A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=30, p=1.0, border_mode=cv2.BORDER_REFLECT_101),
#             # 3. 変換されたパッチの中央から最終サイズをクロップ (64x64)
#             A.CenterCrop(height=final_size, width=final_size),
#         ],
#         additional_targets={"image1": "image",
#                             "image2": "image"},
#         strict=True,
#         seed=137,
#     )
#     
#     original_img_folders = [os.path.join(original_dir, f) for f in [data_folders[k] for k in train_idx]]
#     img_folders = [os.path.join(main_dir, f) for f in [data_folders[k] for k in train_idx]]
#     for i in range(len(original_img_folders)):
#         img_path_list = os.listdir(original_img_folders[i])
#         for j in range(len(img_path_list)):
#             image = tifffile.imread(os.path.join(original_img_folders[i],img_path_list[j]))
#             if preprocess:
#                 for l in range(len(image[0][0])):
#                     image[..., l] = stand_norm(image[..., l], img_minmax[l])
#             phase1, phase2, mito = image[..., 0], image[..., 1], image[..., 2]
#             for k in range(img_n):
#                 augmented = train_aug(image=phase1, image1=phase2, image2=mito)
#                 image_crop = np.stack([augmented['image'], augmented['image1'], augmented['image2']], axis=-1)
#                 base_name, ext = os.path.splitext(img_path_list[j])
#                 new_filename = f"{base_name}_{k}{ext}"
#                 save_path = os.path.join(img_folders[i], new_filename)
#                 tifffile.imwrite(save_path, image_crop)
#         print(f"folder {img_folders[i]} done")
#     
#     if preprocess:
#         original_img_folders = [os.path.join(original_dir, f) for f in [data_folders[k] for k in val_test_idx]]
#         img_folders = [os.path.join(main_dir, f) for f in [data_folders[k] for k in val_test_idx]]
#         for i in range(len(original_img_folders)):
#             img_path_list = os.listdir(original_img_folders[i])
#             for j in range(len(img_path_list)):
#                 image = tifffile.imread(os.path.join(original_img_folders[i],img_path_list[j]))
#                 for l in range(len(image[0][0])):
#                     image[..., l] = stand_norm(image[..., l], img_minmax[l])
#                 tifffile.imwrite(os.path.join(img_folders[i],img_path_list[j]), image)
#             print(f"folder {img_folders[i]} done")
# 
# 
# def produce_images_2(original_dir, preprocess=True, img_n=100, img_size=256, gaussian_crip=False):
#     main_dir = original_dir
#     if preprocess:
#         original_img_folders = [os.path.join(original_dir, f) for f in ['1', '2', '3', '4', '5']]
#         img_list = []
#         for i in range(len(original_img_folders)):
#             img_path_list = os.listdir(original_img_folders[i])
#             for j in range(len(img_path_list)):
#                 image = tifffile.imread(os.path.join(original_img_folders[i], img_path_list[j]))
#                 img = []
#                 for l in range(len(image[0][0])):
#                     img.append(standardization(image[..., l]).flatten())
#                 img_list.append([img])
#         img_list = np.array(img_list)
#         img_minmax = []
#         for l in range(len(image[0][0])):
#             img_concat = np.concatenate(img_list[:, 0, l, :])
#             img_minmax.append([np.percentile(img_concat, 0.1), np.percentile(img_concat, 99.9)])
#         print("norm_measure done")
#         
#     # train_aug = A.Compose(
#     #     [
#     #         A.ShiftScaleRotate(
#     #             shift_limit=0.2, scale_limit=0.2,
#     #             rotate_limit=30, p=0.7,
#     #             border_mode=cv2.BORDER_REFLECT_101
#     #         ),
#     #         A.CropNonEmptyMaskIfExists(
#     #             height=img_size,
#     #             width=img_size,
#     #             p=1.0
#     #         ),
#     #     ],
#     #     additional_targets={"image1": "image",
#     #                         "image2": "image",
#     #                         "image3": "image"},
#     #     strict=True,
#     #     seed=137,
#     # )
#     
#     # 最終的な画像サイズ
#     final_size = img_size
#     # 最初に切り出す、一回り大きいサイズ
#     initial_crop_size = img_size * 2 
#     
#     train_aug = A.Compose(
#         [
#             # 1. 元画像から大きめにクロップ (128x128)
#             A.RandomCrop(height=initial_crop_size, width=initial_crop_size),
#             
#             # 2. そのパッチに対して回転・拡縮
#             #    ※拡縮しすぎると中央に黒い部分が入る可能性があるので、scale_limitは控えめに
#             A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=30, p=1.0),
#             
#             # 3. 変換されたパッチの中央から最終サイズをクロップ (64x64)
#             A.CenterCrop(height=final_size, width=final_size),
#         ],
#         additional_targets={"image1": "image",
#                             "image2": "image"},
#         strict=True,
#         seed=137,
#     )
#     
#     original_img_folders = [os.path.join(original_dir, f) for f in ['1', '2', '3', '4', '5']]
#     img_folders = [os.path.join(main_dir, f) for f in ['c1', 'c2', 'c3', 'c4', 'c5']]
#     for folder in img_folders:
#         os.makedirs(folder, exist_ok=True)
#     for i in range(len(original_img_folders)):
#         img_path_list = os.listdir(original_img_folders[i])
#         for j in tqdm(range(len(img_path_list))):
#             ori_pth = os.path.join(original_img_folders[i],img_path_list[j])
#             image = tifffile.imread(ori_pth)
#             # plt.imshow(image[..., 2])
#             # plt.axis("off")
#             # plt.show()
#             if preprocess:
#                 for l in range(len(image[0][0])):
#                     if l != 2:
#                         image[..., l] = stand_norm(image[..., l], img_minmax[l])
#                     else:
#                         if gaussian_crip:
#                             image[..., 2] = gauss_clip_con(image[..., 2], img_minmax[2])
#                         else:
#                             image[..., 2] = stand_norm(image[..., 2], img_minmax[2])
#             # plt.imshow(image[..., 2])
#             # plt.axis("off")
#             # plt.show()
#             # break
#             phase1, phase2, mito = image[..., 0], image[..., 1], image[..., 2]
#             tifffile.imwrite(ori_pth, image)
#             for k in range(img_n):
#                 augmented = train_aug(image=phase1, image1=phase2, image2=mito)
#                 image_crop = np.stack([augmented['image'], augmented['image1'], augmented['image2']], axis=-1)
#                 base_name, ext = os.path.splitext(img_path_list[j])
#                 new_filename = f"{base_name}_{k}{ext}"
#                 save_path = os.path.join(img_folders[i], new_filename)
#                 tifffile.imwrite(save_path, image_crop)
#         print(f"folder {img_folders[i]} done")